# Assignment 1 — 01: Data Loading & Cleaning

**KQC7016 Data Analytics | Theme 7: Autonomous Driving**  
**Group:** Muhammad Amru Bin Mohamad Sharis (S2116804) & Nor Shahadah Fitrah Binti Ramani (25073210)

## Purpose
Combine the four NHTSA Standing General Order (SGO) AV incident CSV files (ADS current, ADS prior, ADAS current, ADAS prior) into a single cleaned dataframe ready for EDA, statistical analysis, and clustering.

## Source
NHTSA Standing General Order 2021-01, *Incident Reports of Crashes Involving Vehicles Equipped with ADS or Level 2 ADAS*. Public-release dataset.

## Output
`assignment1/notebooks/data/cleaned.csv` — used by `02_eda`, `03_stats`, `04_cluster`.

## 1. Imports & configuration

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

DATA_DIR = Path('../dataset/NHTSA_SGA_AV')
OUT_DIR = Path('data')
OUT_DIR.mkdir(exist_ok=True)

FILES = {
    'ADS_current':   'SGO-2021-01_Incident_Reports_ADS.csv',
    'ADS_prior':     'SGO-2021-01_Incident_Reports_ADS_prior.csv',
    'ADAS_current':  'SGO-2021-01_Incident_Reports_ADAS.csv',
    'ADAS_prior':    'SGO-2021-01_Incident_Reports_ADAS_prior.csv',
}

## 2. Load raw files
Each file loaded with an encoding fallback (one ADAS_prior file is not UTF-8). A `source_file` tag is added so we can audit provenance later.

In [2]:
def load_csv(path: Path) -> pd.DataFrame:
    """Read CSV with utf-8, fall back to latin-1 if needed."""
    try:
        return pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, low_memory=False, encoding='latin-1')

raw = {}
for tag, fname in FILES.items():
    df = load_csv(DATA_DIR / fname)
    df['source_file'] = tag
    raw[tag] = df
    print(f"{tag:15s} rows={len(df):>5d}  cols={df.shape[1]}")

ADS_current     rows=  825  cols=117
ADS_prior       rows= 2295  cols=138
ADAS_current    rows= 1145  cols=117
ADAS_prior      rows= 4027  cols=138


## 3. Schema reconciliation
Current files (2025–2026) use a newer NHTSA schema. Prior files (2019–mid 2025) use the original schema. Their column intersection is 90 (89 schema fields plus our injected `source_file` provenance tag).

A small number of prior-only columns are still analytically valuable for EDA:
- `Lighting`
- `Roadway Surface`
- `Posted Speed Limit (MPH)`
- `Property Damage?`
- `Weather - Fog/Smoke`

We retain these as additional columns; rows from current files will have NaN for them, and any analysis using them will be clearly scoped to the prior-schema subset.

In [3]:
cols_intersection = set(raw['ADS_current'].columns)
for tag in ['ADS_prior', 'ADAS_current', 'ADAS_prior']:
    cols_intersection &= set(raw[tag].columns)

PRIOR_ONLY_KEEP = [
    'Lighting', 'Roadway Surface', 'Posted Speed Limit (MPH)',
    'Property Damage?', 'Weather - Fog/Smoke',
]

keep_cols = sorted(cols_intersection) + PRIOR_ONLY_KEEP
print(f"Intersection columns:        {len(cols_intersection)}")
print(f"Prior-only columns retained: {len(PRIOR_ONLY_KEEP)}")
print(f"Total columns in combined:   {len(keep_cols)}")

Intersection columns:        90
Prior-only columns retained: 5
Total columns in combined:   95


## 4. Concatenate into single dataframe

In [4]:
def select_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Project to keep_cols; add missing prior-only cols as NaN."""
    out = df.copy()
    for c in PRIOR_ONLY_KEEP:
        if c not in out.columns:
            out[c] = np.nan
    return out[keep_cols]

df = pd.concat([select_cols(raw[t]) for t in FILES.keys()], ignore_index=True)
print(f"Combined dataframe: {df.shape[0]} rows x {df.shape[1]} cols")
df['source_file'].value_counts()

Combined dataframe: 8292 rows x 95 cols


source_file
ADAS_prior      4027
ADS_prior       2295
ADAS_current    1145
ADS_current      825
Name: count, dtype: int64

## 5. De-duplication
NHTSA reports are versioned — the same incident can appear multiple times as new information arrives (`Report Version` increments). We keep only the latest version of each `Report ID`.

In [5]:
before = len(df)
df['Report Version'] = pd.to_numeric(df['Report Version'], errors='coerce')
df = (df.sort_values('Report Version', ascending=False)
        .drop_duplicates(subset='Report ID', keep='first')
        .reset_index(drop=True))
print(f"Removed {before - len(df)} duplicate report versions. Rows now: {len(df)}")

Removed 1842 duplicate report versions. Rows now: 6450


## 6. Normalise the `Make` column
Raw values include mixed case (e.g. `Tesla`, `TESLA`, `tesla`). Strip whitespace and upper-case for a clean `Make_clean`.

In [6]:
df['Make_clean'] = (df['Make'].astype(str)
                              .str.strip()
                              .str.upper()
                              .replace({'NAN': np.nan, '': np.nan}))
df['Make_clean'].value_counts().head(15)

Make_clean
TESLA            3214
JAGUAR           1843
CRUISE            296
TOYOTA            181
HYUNDAI            93
HONDA              92
FORD               81
JLR                63
SUBARU             62
PETERBILT          49
CHEVROLET          44
CADILLAC           41
GMC                40
MERCEDES-BENZ      28
NISSAN             27
Name: count, dtype: int64

## 7. Parse `Incident Date` and derive `Year` / `Month`
Raw format is `MMM-YYYY` (e.g. `MAR-2026`). Some rows have `Incident Date - Unknown` flagged.

In [7]:
df['Incident_Date_parsed'] = pd.to_datetime(df['Incident Date'], format='%b-%Y', errors='coerce')
df['Year'] = df['Incident_Date_parsed'].dt.year
df['Month'] = df['Incident_Date_parsed'].dt.month
print(f"Date parsed successfully: {df['Incident_Date_parsed'].notna().sum()} / {len(df)} rows")
print(f"Date range: {df['Incident_Date_parsed'].min()} to {df['Incident_Date_parsed'].max()}")
df['Year'].value_counts().sort_index()

Date parsed successfully: 6430 / 6450 rows
Date range: 2019-08-01 00:00:00 to 2026-03-01 00:00:00


Year
2019.0       2
2020.0      13
2021.0     361
2022.0     816
2023.0    1061
2024.0    1421
2025.0    2232
2026.0     524
Name: count, dtype: int64

## 8. Derive a single `Weather_condition` column
The raw schema uses a one-hot style with `Weather - Clear`, `Weather - Rain`, `Weather - Snow`, `Weather - Cloudy`, `Weather - Severe Wind`, and `Weather - Fog/Smoke` (prior only). Collapse to a single categorical column, picking the first flagged value for multi-weather rows.

In [8]:
WEATHER_FLAGS = {
    'Weather - Clear': 'Clear',
    'Weather - Cloudy': 'Cloudy',
    'Weather - Rain': 'Rain',
    'Weather - Snow': 'Snow',
    'Weather - Severe Wind': 'Severe Wind',
    'Weather - Fog/Smoke': 'Fog/Smoke',
}

def derive_weather(row):
    for col, label in WEATHER_FLAGS.items():
        val = row.get(col)
        if isinstance(val, str) and val.strip().upper() == 'Y':
            return label
    return np.nan

df['Weather_condition'] = df.apply(derive_weather, axis=1)
df['Weather_condition'].value_counts(dropna=False)

Weather_condition
Clear        3664
NaN          1383
Cloudy       1195
Rain          188
Fog/Smoke      11
Snow            9
Name: count, dtype: int64

## 9. Ordinal encoding of injury severity
`Highest Injury Severity Alleged` is ordinal. Encode for the Mann-Whitney U test in `03_stats`.

NHTSA uses several label variants for the same severity tier (e.g. `Property Damage. No Injured Reported` = no injury; `Minor W/ Hospitalization` and `Minor W/O Hospitalization` both = minor). We collapse all variants of the same tier to a single code so the known-severity subset is not artificially shrunk by label drift.

| Severity tier | Raw labels collapsed | Code |
|---|---|---|
| No Injury | No Injuries Reported, No Injured Reported, No Injury, Property Damage. No Injured Reported | 0 |
| Minor | Minor, Minor W/O Hospitalization, Minor W/ Hospitalization | 1 |
| Moderate | Moderate, Moderate W/O Hospitalization, Moderate W/ Hospitalization | 2 |
| Serious | Serious, Serious W/O Hospitalization, Serious W/ Hospitalization | 3 |
| Fatality | Fatality | 4 |
| Unknown | Unknown | NaN |

In [9]:
print('Raw severity values:')
print(df['Highest Injury Severity Alleged'].value_counts(dropna=False))

Raw severity values:
Highest Injury Severity Alleged
Unknown                                 3173
No Injuries Reported                    1937
Property Damage. No Injured Reported     679
Minor                                    225
No Injured Reported                       84
Minor W/O Hospitalization                 82
Moderate                                  77
Fatality                                  69
Serious                                   56
Minor W/ Hospitalization                  45
Moderate W/ Hospitalization               12
Serious W/ Hospitalization                 8
Moderate W/O Hospitalization               2
Serious W/O Hospitalization                1
Name: count, dtype: int64


In [10]:
SEVERITY_MAP = {
    'No Injuries Reported': 0,
    'No Injured Reported': 0,
    'No Injury': 0,
    'Property Damage. No Injured Reported': 0,
    'Minor': 1,
    'Minor W/O Hospitalization': 1,
    'Minor W/ Hospitalization': 1,
    'Moderate': 2,
    'Moderate W/O Hospitalization': 2,
    'Moderate W/ Hospitalization': 2,
    'Serious': 3,
    'Serious W/O Hospitalization': 3,
    'Serious W/ Hospitalization': 3,
    'Fatality': 4,
}

def encode_severity(val):
    if not isinstance(val, str):
        return np.nan
    return SEVERITY_MAP.get(val.strip(), np.nan)

df['Severity_num'] = df['Highest Injury Severity Alleged'].apply(encode_severity)
df['Severity_num'].value_counts(dropna=False).sort_index()

Severity_num
0.0    2700
1.0     352
2.0      91
3.0      65
4.0      69
NaN    3173
Name: count, dtype: int64

## 10. System-type column (audit)
`Automation System Engaged?` should be ADS / ADAS / Unknown. Confirm and create `System_type` for convenience.

In [11]:
df['System_type'] = df['Automation System Engaged?'].astype(str).str.strip().str.upper()
df['System_type'] = df['System_type'].replace({'NAN': 'Unknown'})
df['System_type'].value_counts(dropna=False)

System_type
ADAS                      3549
ADS                       2670
UNKNOWN, SEE NARRATIVE     231
Name: count, dtype: int64

## 11. Numeric coercion for speed columns

In [12]:
for col in ['SV Precrash Speed (MPH)', 'Posted Speed Limit (MPH)', 'Model Year']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df[['SV Precrash Speed (MPH)', 'Posted Speed Limit (MPH)', 'Model Year']].describe()

,SV Precrash Speed (MPH),Posted Speed Limit (MPH),Model Year
count,5912.000000,4060.000000,6436.000000
mean,26.838633,43.059852,2022.133002
std,30.396495,18.050601,2.255690
min,0.000000,0.000000,2014.000000
25%,0.000000,25.000000,2021.000000
50%,20.000000,40.000000,2023.000000
75%,48.000000,65.000000,2024.000000
max,925.000000,80.000000,2026.000000


## 12. Missing-value summary
Report top-missing analytical columns. This feeds the Data Understanding & Preparation section of the report.

In [13]:
analysis_cols = [
    'System_type', 'Make_clean', 'Year', 'Roadway Type', 'Crash With',
    'Weather_condition', 'Lighting', 'Roadway Surface', 'Property Damage?',
    'SV Precrash Speed (MPH)', 'Posted Speed Limit (MPH)',
    'Highest Injury Severity Alleged', 'Severity_num', 'State',
]
miss = (df[analysis_cols].isna().sum() / len(df) * 100).round(1).sort_values(ascending=False)
miss.to_frame('missing_%')

,missing_%
Severity_num,49.2
Posted Speed Limit (MPH),37.1
Lighting,29.1
Roadway Surface,29.1
Property Damage?,29.1
Weather_condition,21.4
SV Precrash Speed (MPH),8.3
State,0.5
Year,0.3
System_type,0.0


## 13. Final shape summary

In [14]:
print(f"Final cleaned dataframe: {df.shape[0]} rows x {df.shape[1]} cols")
print(f"\nDate range: {df['Incident_Date_parsed'].min().date()} to {df['Incident_Date_parsed'].max().date()}")
print(f"\nSystem type breakdown:")
print(df['System_type'].value_counts())
print(f"\nTop manufacturers:")
print(df['Make_clean'].value_counts().head(8))

Final cleaned dataframe: 6450 rows x 102 cols

Date range: 2019-08-01 to 2026-03-01

System type breakdown:
System_type
ADAS                      3549
ADS                       2670
UNKNOWN, SEE NARRATIVE     231
Name: count, dtype: int64

Top manufacturers:
Make_clean
TESLA      3214
JAGUAR     1843
CRUISE      296
TOYOTA      181
HYUNDAI      93
HONDA        92
FORD         81
JLR          63
Name: count, dtype: int64


## 14. Export cleaned dataset

In [15]:
OUT_PATH = OUT_DIR / 'cleaned.csv'
df.to_csv(OUT_PATH, index=False)
print(f"Wrote: {OUT_PATH.resolve()}")
print(f"Size : {OUT_PATH.stat().st_size / 1024:.1f} KB")

Wrote: /home/lilmru/Masters/data_analytics/github_repo/assignment1/notebooks/data/cleaned.csv
Size : 7135.0 KB


---
## Cleaning summary

| Step | Action |
|------|--------|
| 1 | Loaded 4 CSVs (ADS/ADAS × current/prior) with encoding fallback |
| 2 | Reconciled current vs prior schemas → 90-column intersection (89 schema + `source_file`) + 5 prior-only analytical cols |
| 3 | Deduplicated by `Report ID`, keeping latest `Report Version` |
| 4 | Normalised `Make` → `Make_clean` (strip + upper) |
| 5 | Parsed `Incident Date` → `Year`, `Month` |
| 6 | Collapsed binary weather flags → single `Weather_condition` |
| 7 | Ordinal-encoded injury severity → `Severity_num` (0–4), collapsing W/Hospitalization and Property-Damage label variants |
| 8 | Coerced numeric speed/year cols |
| 9 | Reported missingness on analysis-relevant cols |
| 10 | Exported `data/cleaned.csv` for downstream notebooks |

Next: `02_eda.ipynb` for visualisations.